[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/8-evals.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 8. Evals

Five live cases and one canned failure. The grader looks at phrases and at the tool log. A confident tone is not a grade.

Support: an opened jar, sesame crunch, a cracked jar. Buying: chili oil to Ohio, fresh labneh to Ohio. The canned row invents a cash refund and a password. It does not call the model. It stays in the set because it fails.

The cases live in `tutorial.common.evals`. The topic tool, the catalog, and the loop are imported so this notebook runs alone.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## Grade the set

A pass needs the phrases and the tools. The canned row is supposed to fail.

In [ ]:
import tutorial.common.facts
import tutorial.common.catalog

from tutorial.common.evals import CASES, grade_case, print_grade
from tutorial.common.get_db import reset_db
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent

reset_db()
passed = 0
for case in CASES:
    print("---", case["id"], case["kind"], "---")
    if case["live"]:
        result = run_agent(case["question"], system_text())
        text = result["text"]
        tool_log = result["tool_log"]
        print("ANSWER:", text)
    else:
        text = case["text"]
        tool_log = case["tool_log"]
        print("CANNED:", text)
    failures = grade_case(case, text, tool_log)
    if print_grade(case["id"], failures):
        passed += 1
print("passed", passed, "of", len(CASES))
check(len(CASES) >= 6, "the set is more than a three-row toy")
check(any(not case["live"] for case in CASES), "a canned known failure stays in the set")
check(passed == len(CASES) - 1, "the canned failure is the only miss")